# Reversible Circuits Project: Phase 2

Canonical form of a reversible circuit, from Section IV-A of Feng and Li
(*A Complete Set of Transformation Rules for Reversible Circuits*, IEEE TCAD, 2026).

**How to run:** choose **Runtime > Run all** from the menu. The whole notebook takes under a minute.

| Step | What it does |
|---|---|
| 1 | Creates the Phase 1 files `revcircuit.py` and `rules.py` (unchanged) |
| 2 | Creates `canonical.py`: Hamiltonian path, the gate set Δ_H, Definition 1, Algorithm 1 |
| 3 | Creates `verify_phase2.py`, the Phase 2 tests |
| 4 | Runs the tests |
| 5 | A small example you can edit and re-run |

This notebook is self-contained. You do not need to run the Phase 1 notebook first.

## Step 1: Phase 1 files (unchanged)

In [ ]:
%%writefile revcircuit.py
"""Reversible circuit model and simulator (Phase 1).

Gates are mixed-polarity multiple-control Toffoli (MPMCT) gates, written
G[P, N, q] as in Feng and Li, "A Complete Set of Transformation Rules for
Reversible Circuits", IEEE TCAD, 2026:

    P  set of positive control lines
    N  set of negative control lines
    q  target line

The gate flips q when every line in P is 1 and every line in N is 0.
Special cases: X[q] = G[{}, {}, q], CNOT[p, q] = G[{p}, {}, q], and a
Toffoli gate is G[{a, b}, {}, q].

A basis state is stored as an integer whose bit i is the value of line q_i.
A circuit is a list of gates; the leftmost gate executes first.
"""
from __future__ import annotations

import random
from dataclasses import dataclass
from itertools import product
from typing import Iterable, Iterator, List, Optional, Sequence, Tuple


# ---------------------------------------------------------------------------
# Gate
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class Gate:
    """One MPMCT gate G[pos, neg, target]."""

    pos: frozenset
    neg: frozenset
    target: int

    def __post_init__(self) -> None:
        object.__setattr__(self, "pos", frozenset(self.pos))
        object.__setattr__(self, "neg", frozenset(self.neg))
        if self.pos & self.neg:
            raise ValueError("a line cannot be both a positive and a negative control")
        if self.target in self.pos or self.target in self.neg:
            raise ValueError("the target line cannot also be a control")
        if min(self.lines) < 0:
            raise ValueError("line numbers must be non-negative")
        # Bit masks used by the simulator.
        object.__setattr__(self, "_pmask", sum(1 << l for l in self.pos))
        object.__setattr__(self, "_nmask", sum(1 << l for l in self.neg))
        object.__setattr__(self, "_tmask", 1 << self.target)

    @property
    def controls(self) -> frozenset:
        return self.pos | self.neg

    @property
    def lines(self) -> frozenset:
        return self.pos | self.neg | {self.target}

    def apply(self, state: int) -> int:
        """Return the state after this gate acts on `state`."""
        if (state & self._pmask) == self._pmask and not (state & self._nmask):
            return state ^ self._tmask
        return state

    def __str__(self) -> str:
        t = self.target
        if not self.neg:
            if not self.pos:
                return f"X[q{t}]"
            if len(self.pos) == 1:
                (c,) = self.pos
                return f"CNOT[q{c},q{t}]"

        def fmt(s: frozenset) -> str:
            return "{" + ",".join(f"q{l}" for l in sorted(s)) + "}"

        return f"G[{fmt(self.pos)},{fmt(self.neg)},q{t}]"


def G(pos: Iterable[int], neg: Iterable[int], target: int) -> Gate:
    """General gate G[P, N, q]."""
    return Gate(frozenset(pos), frozenset(neg), target)


def X(q: int) -> Gate:
    """NOT gate on line q."""
    return Gate(frozenset(), frozenset(), q)


def CNOT(control: int, target: int) -> Gate:
    """Controlled-NOT (Feynman gate)."""
    return Gate(frozenset({control}), frozenset(), target)


def TOF(c1: int, c2: int, target: int) -> Gate:
    """Toffoli gate with two positive controls."""
    return Gate(frozenset({c1, c2}), frozenset(), target)


def MCT(controls: Iterable[int], target: int) -> Gate:
    """Multiple-control Toffoli gate with positive controls only."""
    return Gate(frozenset(controls), frozenset(), target)


# ---------------------------------------------------------------------------
# Circuit
# ---------------------------------------------------------------------------
class Circuit:
    """A reversible circuit on n lines: an ordered list of gates."""

    def __init__(self, n: int, gates: Iterable[Gate] = ()) -> None:
        self.n = n
        self.gates: List[Gate] = list(gates)
        for g in self.gates:
            if max(g.lines) >= n:
                raise ValueError(f"gate {g} uses a line outside q0..q{n - 1}")

    # -- container behaviour ------------------------------------------------
    def __len__(self) -> int:
        return len(self.gates)

    def __getitem__(self, i: int) -> Gate:
        return self.gates[i]

    def __iter__(self) -> Iterator[Gate]:
        return iter(self.gates)

    def __eq__(self, other: object) -> bool:
        """Structural equality: same lines and the same gate list."""
        return isinstance(other, Circuit) and self.n == other.n and self.gates == other.gates

    def __add__(self, other: "Circuit") -> "Circuit":
        """Concatenation: self executes first, then other."""
        if self.n != other.n:
            raise ValueError("circuits must have the same number of lines")
        return Circuit(self.n, self.gates + other.gates)

    def __str__(self) -> str:
        return " ".join(str(g) for g in self.gates) if self.gates else "(empty)"

    def copy(self) -> "Circuit":
        return Circuit(self.n, self.gates)

    def replace(self, i: int, j: int, new_gates: Sequence[Gate]) -> "Circuit":
        """Return a new circuit with gates[i:j] replaced by new_gates."""
        return Circuit(self.n, self.gates[:i] + list(new_gates) + self.gates[j:])

    def inverse(self) -> "Circuit":
        """Every gate is its own inverse, so the inverse is the reversed list."""
        return Circuit(self.n, reversed(self.gates))

    # -- simulation ---------------------------------------------------------
    def apply(self, state: int) -> int:
        for g in self.gates:
            state = g.apply(state)
        return state

    def permutation(self) -> Tuple[int, ...]:
        """The reversible function computed, as a tuple over all 2^n states."""
        return tuple(self.apply(s) for s in range(1 << self.n))

    def equivalent(self, other: "Circuit") -> bool:
        """Functional equivalence: both circuits compute the same function."""
        return self.n == other.n and self.permutation() == other.permutation()

    # -- display ------------------------------------------------------------
    def _bits(self, state: int) -> str:
        return " ".join(str((state >> i) & 1) for i in range(self.n))

    def truth_table(self) -> str:
        head = " ".join(f"q{i}" for i in range(self.n))
        rows = [f"{head} | {head}"]
        for s, t in enumerate(self.permutation()):
            pad = "  ".join(self._bits(s).split())
            out = "  ".join(self._bits(t).split())
            rows.append(f" {pad} |  {out}")
        return "\n".join(rows)

    def draw(self) -> str:
        """Text drawing: circled plus = target, filled dot = positive control, open dot = negative control."""
        width = len(f"q{self.n - 1}")
        rows = []
        for r in range(self.n):
            cells = []
            for g in self.gates:
                lo, hi = min(g.lines), max(g.lines)
                if r == g.target:
                    ch = "\u2295"  # target
                elif r in g.pos:
                    ch = "\u25cf"  # filled dot
                elif r in g.neg:
                    ch = "\u25cb"  # open dot
                elif lo < r < hi:
                    ch = "\u253c"  # wire crossing a gate
                else:
                    ch = "\u2500"
                cells.append("\u2500" + ch + "\u2500")
            rows.append(f"q{r}:".ljust(width + 2) + "\u2500" + "".join(cells) + "\u2500")
        return "\n".join(rows)


# ---------------------------------------------------------------------------
# Helpers for experiments
# ---------------------------------------------------------------------------
def pn_choices(lines: Sequence[int]) -> Iterator[Tuple[frozenset, frozenset]]:
    """All ways to make each given line absent, a positive or a negative control."""
    for states in product((0, 1, 2), repeat=len(lines)):
        pos = frozenset(l for l, s in zip(lines, states) if s == 1)
        neg = frozenset(l for l, s in zip(lines, states) if s == 2)
        yield pos, neg


def all_gates(n: int) -> List[Gate]:
    """Every MPMCT gate on n lines: n * 3^(n-1) gates."""
    gates = []
    for target in range(n):
        others = [l for l in range(n) if l != target]
        for pos, neg in pn_choices(others):
            gates.append(Gate(pos, neg, target))
    return gates


def random_gate(n: int, rng: Optional[random.Random] = None) -> Gate:
    rng = rng or random
    target = rng.randrange(n)
    pos, neg = set(), set()
    for l in range(n):
        if l == target:
            continue
        s = rng.randrange(3)
        if s == 1:
            pos.add(l)
        elif s == 2:
            neg.add(l)
    return Gate(frozenset(pos), frozenset(neg), target)


def random_circuit(n: int, length: int, rng: Optional[random.Random] = None) -> Circuit:
    return Circuit(n, [random_gate(n, rng) for _ in range(length)])

In [ ]:
%%writefile rules.py
"""Transformation rules for reversible circuits (Phase 1).

Rules 1 to 5 are the complete rule set RC of Feng and Li (IEEE TCAD, 2026).
Rules 6 to 10 are the rules the paper derives from RC.

Every rewrite function takes a circuit and a position i and returns a NEW
circuit with the rule applied at position i, or None when the rule does not
match there. The input circuit is never modified.

For rules stated on gates rather than positions, a `ruleK_gates` or
`ruleK_sides` helper returns the gate lists on each side of the rule.
"""
from __future__ import annotations

from itertools import combinations
from typing import Iterable, List, Optional, Sequence, Tuple

from revcircuit import CNOT, Circuit, G, Gate, X


# ===========================================================================
# Rule 1:  A A  ==  (empty)
# ===========================================================================
def rule1_cancel(c: Circuit, i: int) -> Optional[Circuit]:
    """Delete two adjacent identical gates at positions i and i+1."""
    if 0 <= i and i + 1 < len(c) and c[i] == c[i + 1]:
        return c.replace(i, i + 2, [])
    return None


def rule1_insert(c: Circuit, i: int, gate: Gate) -> Circuit:
    """Rule 1 read right to left: insert the pair `gate gate` before position i."""
    return c.replace(i, i, [gate, gate])


# ===========================================================================
# Rule 2:  G[P, N+{p}, q] G[P+{p}, N, q]  ==  G[P, N, q]
# ===========================================================================
def rule2_gates(gate: Gate, p: int) -> List[Gate]:
    """The pair A0, A1 that Rule 2 merges into `gate`, using extra line p."""
    if p in gate.lines:
        raise ValueError("p must be a line the gate does not use")
    return [G(gate.pos, gate.neg | {p}, gate.target), G(gate.pos | {p}, gate.neg, gate.target)]


def rule2_merge(c: Circuit, i: int) -> Optional[Circuit]:
    """Merge gates i and i+1 when they are A0 A1 in the order of the paper."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    extra = a1.pos - a0.pos
    if a0.target != a1.target or len(extra) != 1:
        return None
    (p,) = extra
    merged = G(a1.pos - {p}, a1.neg, a1.target)
    if rule2_gates(merged, p) == [a0, a1]:
        return c.replace(i, i + 2, [merged])
    return None


def rule2_split(c: Circuit, i: int, p: int) -> Optional[Circuit]:
    """Rule 2 read right to left: split gate i into A0 A1 on unused line p."""
    if not (0 <= i < len(c)) or not (0 <= p < c.n) or p in c[i].lines:
        return None
    return c.replace(i, i + 1, rule2_gates(c[i], p))


# ===========================================================================
# Rule 3:  A B == B A  when a control has opposite polarity in A and B
# ===========================================================================
def rule3_condition(a: Gate, b: Gate) -> bool:
    return bool(a.pos & b.neg) or bool(b.pos & a.neg)


def rule3_commute(c: Circuit, i: int) -> Optional[Circuit]:
    """Swap gates i and i+1 when they share a control of opposite polarity."""
    if 0 <= i and i + 1 < len(c) and rule3_condition(c[i], c[i + 1]):
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# ===========================================================================
# Rule 4:  A B A C1 == C2 A B A,  A = CNOT[p,q], B = CNOT[q,p]
#          (moving a swap of p and q through a gate)
# ===========================================================================
def rule4_sides(p: int, q: int, P: Iterable[int], N: Iterable[int],
                negative: bool = False) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 4.

    negative=False is condition (a): C1 has q as positive control, target p.
    negative=True  is condition (b): C1 has q as negative control, target p.
    """
    P, N = frozenset(P), frozenset(N)
    a, b = CNOT(p, q), CNOT(q, p)
    if negative:
        c1, c2 = G(P, N | {q}, p), G(P, N | {p}, q)
    else:
        c1, c2 = G(P | {q}, N, p), G(P | {p}, N, q)
    return [a, b, a, c1], [c2, a, b, a]


def _is_cnot(g: Gate) -> bool:
    return len(g.pos) == 1 and not g.neg


def rule4_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Apply Rule 4 to the four gates starting at i, in either direction."""
    if i < 0 or i + 4 > len(c):
        return None
    seg = c.gates[i:i + 4]
    # Left to right:  A B A C1  ->  C2 A B A
    if _is_cnot(seg[0]):
        (p,) = seg[0].pos
        q = seg[0].target
        c1 = seg[3]
        if c1.target == p and q in c1.controls:
            lhs, rhs = rule4_sides(p, q, c1.pos - {q}, c1.neg - {q}, q in c1.neg)
            if seg == lhs:
                return c.replace(i, i + 4, rhs)
    # Right to left:  C2 A B A  ->  A B A C1
    if _is_cnot(seg[1]):
        (p,) = seg[1].pos
        q = seg[1].target
        c2 = seg[0]
        if c2.target == q and p in c2.controls:
            lhs, rhs = rule4_sides(p, q, c2.pos - {p}, c2.neg - {p}, p in c2.neg)
            if seg == rhs:
                return c.replace(i, i + 4, lhs)
    return None


# ===========================================================================
# Rule 5:  A0 A1 B1..Bm..B1 A1 A0  ==  B'1..B'm..B'1
#          (changes the polarity of control q0)
# ===========================================================================
def rule5_sides(P: Iterable[int], N: Iterable[int], Q: Sequence[int],
                q0: int) -> Tuple[List[Gate], List[Gate]]:
    """Both sides of Rule 5 for control sets P, N, ordered list Q and line q0."""
    P, N, Q = frozenset(P), frozenset(N), list(Q)
    Qs = frozenset(Q)
    if not Q or len(Qs) != len(Q):
        raise ValueError("Q must be a non-empty list of distinct lines")
    a0 = G(P, N | Qs, q0)
    a1 = G(P | Qs, N, q0)
    b = [G(P | {q0} | frozenset(Q[k + 1:]), N | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    bp = [G(P | frozenset(Q[k + 1:]), N | {q0} | frozenset(Q[:k]), Q[k]) for k in range(len(Q))]
    lhs = [a0, a1] + b + b[-2::-1] + [a1, a0]
    rhs = bp + bp[-2::-1]
    return lhs, rhs


def rule5_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Left to right: replace the 2m+3 gate pattern starting at i by 2m-1 gates."""
    if i < 0 or i + 1 >= len(c):
        return None
    a0, a1 = c[i], c[i + 1]
    Qs = a1.pos - a0.pos
    if a0.target != a1.target or not Qs or a0.neg - a1.neg != Qs:
        return None
    if a0.pos - a1.pos or a1.neg - a0.neg:
        return None
    m = len(Qs)
    end = i + 2 * m + 3
    if end > len(c):
        return None
    order = [g.target for g in c.gates[i + 2:i + 2 + m]]
    if frozenset(order) != Qs:
        return None
    lhs, rhs = rule5_sides(a0.pos, a1.neg, order, a0.target)
    if c.gates[i:end] == lhs:
        return c.replace(i, end, rhs)
    return None


def rule5_reverse(c: Circuit, i: int, Q: Sequence[int], q0: int) -> Optional[Circuit]:
    """Right to left: expand B'1..B'm..B'1 at i, given the list Q and line q0."""
    Q = list(Q)
    end = i + 2 * len(Q) - 1
    if i < 0 or not Q or end > len(c):
        return None
    b1 = c[i]
    if b1.target != Q[0] or q0 not in b1.neg:
        return None
    try:
        lhs, rhs = rule5_sides(b1.pos - frozenset(Q[1:]), b1.neg - {q0}, Q, q0)
    except ValueError:
        return None
    if c.gates[i:end] == rhs:
        return c.replace(i, end, lhs)
    return None


# ===========================================================================
# Derived rules 6 to 10
# ===========================================================================
# Rule 6:  G[P,N,q] == product over all subsets S of Q of G[P+S, N+(Q-S), q]
def rule6_gates(gate: Gate, Q: Iterable[int]) -> List[Gate]:
    Q = sorted(set(Q))
    if set(Q) & gate.lines:
        raise ValueError("Q must not overlap the lines of the gate")
    out = []
    for r in range(len(Q) + 1):
        for sub in combinations(Q, r):
            out.append(G(gate.pos | set(sub), gate.neg | (set(Q) - set(sub)), gate.target))
    return out


def rule6_expand(c: Circuit, i: int, Q: Iterable[int]) -> Optional[Circuit]:
    """Expand gate i over all polarity patterns of the unused lines Q."""
    Q = set(Q)
    if not (0 <= i < len(c)) or not Q or Q & c[i].lines or max(Q) >= c.n or min(Q) < 0:
        return None
    return c.replace(i, i + 1, rule6_gates(c[i], Q))


# Rule 7:  two gates with the same target commute
def rule7_commute(c: Circuit, i: int) -> Optional[Circuit]:
    if 0 <= i and i + 1 < len(c) and c[i].target == c[i + 1].target:
        return c.replace(i, i + 2, [c[i + 1], c[i]])
    return None


# Rule 8:  G[P,N,q] == X[N] G[P+N, {}, q] X[N]
def rule8_gates(gate: Gate) -> List[Gate]:
    xs = [X(l) for l in sorted(gate.neg)]
    return xs + [G(gate.pos | gate.neg, (), gate.target)] + xs


def rule8_expand(c: Circuit, i: int) -> Optional[Circuit]:
    """Replace the negative controls of gate i by X gates before and after."""
    if 0 <= i < len(c) and c[i].neg:
        return c.replace(i, i + 1, rule8_gates(c[i]))
    return None


def rule8_collapse(c: Circuit, i: int) -> Optional[Circuit]:
    """Right to left: absorb X..X B X..X starting at i into one gate."""
    k = 0
    while i + k < len(c) and not c[i + k].controls:
        k += 1
    if i < 0 or k == 0 or i + 2 * k + 1 > len(c):
        return None
    b = c[i + k]
    lines = frozenset(g.target for g in c.gates[i:i + k])
    if len(lines) != k or b.neg or not lines <= b.pos:
        return None
    merged = G(b.pos - lines, lines, b.target)
    if c.gates[i:i + 2 * k + 1] == rule8_gates(merged):
        return c.replace(i, i + 2 * k + 1, [merged])
    return None


# Rule 9:  A B1 == B2 A, where A targets p and B1, B2 differ in the polarity of p
def _rule9_pair(a: Gate, b: Gate) -> Optional[Gate]:
    """If A can pass B by Rule 9, return B with the polarity of A's target flipped."""
    p = a.target
    if p not in b.controls:
        return None
    if not (a.pos <= b.pos - {p} and a.neg <= b.neg - {p}):
        return None
    if p in b.pos:
        return G(b.pos - {p}, b.neg | {p}, b.target)
    return G(b.pos | {p}, b.neg - {p}, b.target)


def rule9_apply(c: Circuit, i: int) -> Optional[Circuit]:
    """Move a gate past its neighbour, flipping one control polarity in the neighbour."""
    if i < 0 or i + 1 >= len(c):
        return None
    g1, g2 = c[i], c[i + 1]
    flipped = _rule9_pair(g1, g2)          # A B1 -> B2 A
    if flipped is not None:
        return c.replace(i, i + 2, [flipped, g1])
    flipped = _rule9_pair(g2, g1)          # B2 A -> A B1
    if flipped is not None:
        return c.replace(i, i + 2, [g2, flipped])
    return None


# Rule 10:  A B A == B A B,  A = G[P+{p}, {}, q],  B = G[P+{q}, {}, p]
def rule10_sides(P: Iterable[int], p: int, q: int) -> Tuple[List[Gate], List[Gate]]:
    P = frozenset(P)
    a, b = G(P | {p}, (), q), G(P | {q}, (), p)
    return [a, b, a], [b, a, b]


def rule10_apply(c: Circuit, i: int) -> Optional[Circuit]:
    if i < 0 or i + 3 > len(c):
        return None
    a, b = c[i], c[i + 1]
    p, q = b.target, a.target
    if a.neg or b.neg or p == q or p not in a.pos or q not in b.pos:
        return None
    if a.pos - {p} != b.pos - {q}:
        return None
    lhs, rhs = rule10_sides(a.pos - {p}, p, q)
    if c.gates[i:i + 3] == lhs:
        return c.replace(i, i + 3, rhs)
    return None

## Step 2: Canonical form

The main ideas in `canonical.py`:

- **Hamiltonian path.** All 2^n bit strings are listed so that neighbours differ in one bit. The default is the Gray code, which for 3 lines is the path of Example 4 in the paper.
- **Gate set Δ_H.** Gate `M_i` exchanges the neighbouring strings `a_i` and `a_(i+1)`. It is an n-bit gate: one target and n − 1 controls.
- **Algorithm 1.** A reversible function rearranges the list. The algorithm puts each string in place, from position 0 upwards, by moving it down with the gates `M_i M_(i+1) ... M_(l-1)`.

In [ ]:
%%writefile canonical.py
"""Canonical form of a reversible circuit (Phase 2).

Implements Section IV-A of Feng and Li, "A Complete Set of Transformation
Rules for Reversible Circuits", IEEE TCAD, 2026:

  * a Hamiltonian path H = (a_0, ..., a_{2^n - 1}) of the n-hypercube,
  * the gate set Delta_H = {M_0, ..., M_{2^n - 2}}, where the n-bit gate M_i
    exchanges the neighbouring strings a_i and a_{i+1},
  * Definition 1 (canonical form) and Fact 1,
  * Algorithm 1, which builds the canonical circuit of a reversible function.

Strings follow the paper: the string "s0 s1 ... s(n-1)" gives line q_i the
value s_i. Inside the program a state is an integer whose bit i is s_i, the
same convention as revcircuit.py.

Circuits are written left to right in execution order, as in the paper. The
canonical circuit C = C_m ... C_1 C_0 therefore runs block C_m first.
"""
from __future__ import annotations

from dataclasses import dataclass
from math import factorial
from typing import Dict, List, Optional, Sequence, Tuple

from revcircuit import Circuit, G, Gate


# ---------------------------------------------------------------------------
# Strings and Hamiltonian paths
# ---------------------------------------------------------------------------
def state_of(bits: str) -> int:
    """Convert a paper-style string, e.g. "010" (q0=0, q1=1, q2=0), to a state."""
    return sum(1 << i for i, ch in enumerate(bits) if ch == "1")


def bits_of(state: int, n: int) -> str:
    """Convert a state back to a paper-style string of length n."""
    return "".join(str((state >> i) & 1) for i in range(n))


def gray_path(n: int) -> List[int]:
    """Reflected Gray code as a Hamiltonian path of the n-hypercube.

    For n = 3 this is the path of Example 4 in the paper:
    000, 001, 011, 010, 110, 111, 101, 100.
    """
    return [state_of(format(k ^ (k >> 1), f"0{n}b")) for k in range(1 << n)]


def is_hamiltonian_path(n: int, path: Sequence[int]) -> bool:
    """True when `path` visits every n-bit state once and each step flips one bit."""
    if sorted(path) != list(range(1 << n)):
        return False
    return all(bin(a ^ b).count("1") == 1 for a, b in zip(path, path[1:]))


def _resolve_path(n: int, path: Optional[Sequence[int]]) -> List[int]:
    if path is None:
        return gray_path(n)
    path = list(path)
    if not is_hamiltonian_path(n, path):
        raise ValueError("path is not a Hamiltonian path of the n-hypercube")
    return path


# ---------------------------------------------------------------------------
# The gate set Delta_H
# ---------------------------------------------------------------------------
def exchange_gate(a: int, b: int, n: int) -> Gate:
    """The n-bit gate that exchanges states a and b, which differ in one bit.

    Every other line is a control: positive where a and b hold 1, negative
    where they hold 0. All other states are left unchanged.
    """
    diff = a ^ b
    if diff == 0 or diff & (diff - 1):
        raise ValueError("the two states must differ in exactly one bit")
    target = diff.bit_length() - 1
    pos = [l for l in range(n) if l != target and (a >> l) & 1]
    neg = [l for l in range(n) if l != target and not (a >> l) & 1]
    return G(pos, neg, target)


def delta_gates(n: int, path: Optional[Sequence[int]] = None) -> List[Gate]:
    """Delta_H = [M_0, ..., M_{2^n - 2}] for the given path (Gray code by default)."""
    path = _resolve_path(n, path)
    return [exchange_gate(path[i], path[i + 1], n) for i in range(len(path) - 1)]


def circuit_from_indices(n: int, indices: Sequence[int],
                         path: Optional[Sequence[int]] = None) -> Circuit:
    """The circuit M_i1 M_i2 ... for a list of gate indices."""
    delta = delta_gates(n, path)
    return Circuit(n, [delta[i] for i in indices])


def indices_of_circuit(circuit: Circuit, path: Optional[Sequence[int]] = None) -> List[int]:
    """Indices of a circuit whose gates all belong to Delta_H."""
    lookup: Dict[Gate, int] = {g: i for i, g in enumerate(delta_gates(circuit.n, path))}
    try:
        return [lookup[g] for g in circuit]
    except KeyError as err:
        raise ValueError(f"gate {err.args[0]} is not in Delta_H") from None


# ---------------------------------------------------------------------------
# Definition 1 and Fact 1
# ---------------------------------------------------------------------------
def split_blocks(indices: Sequence[int]) -> List[List[int]]:
    """Cut an index list into maximal runs i, i+1, i+2, ... (execution order)."""
    blocks: List[List[int]] = []
    for i in indices:
        if blocks and blocks[-1][-1] + 1 == i:
            blocks[-1].append(i)
        else:
            blocks.append([i])
    return blocks


def is_canonical(indices: Sequence[int], n: int) -> bool:
    """Definition 1: blocks of consecutive gates whose first indices decrease.

    In execution order the circuit is C_m ... C_1 C_0, and the first index of
    C_i must be smaller than the first index of C_j whenever i < j.
    """
    if any(not 0 <= i <= (1 << n) - 2 for i in indices):
        return False
    starts = [block[0] for block in split_blocks(indices)]
    return all(later < earlier for earlier, later in zip(starts, starts[1:]))


def fact1_holds(indices: Sequence[int], n: int) -> bool:
    """Fact 1: at most 2^n - 1 blocks, and M_i occurs at most i + 1 times."""
    if len(split_blocks(indices)) > (1 << n) - 1:
        return False
    return all(indices.count(i) <= i + 1 for i in set(indices))


def count_canonical_forms(n: int) -> int:
    """Number of index lists that satisfy Definition 1.

    For each first index x there is either no block or a block of length
    1 .. 2^n - 1 - x, giving 2^n - x choices. The product is (2^n)!, the
    number of n-bit reversible functions.
    """
    total = 1
    for x in range((1 << n) - 1):
        total *= (1 << n) - x
    assert total == factorial(1 << n)
    return total


def max_canonical_gates(n: int) -> int:
    """Largest possible canonical circuit: every block has full length."""
    size = 1 << n
    return size * (size - 1) // 2


# ---------------------------------------------------------------------------
# Algorithm 1
# ---------------------------------------------------------------------------
@dataclass(frozen=True)
class CanonicalForm:
    """Canonical circuit of a reversible function, based on a Hamiltonian path."""

    n: int
    path: Tuple[int, ...]
    blocks: Tuple[Tuple[int, ...], ...]      # execution order: C_m, ..., C_1, C_0

    @property
    def indices(self) -> List[int]:
        """Gate indices in execution order."""
        return [i for block in self.blocks for i in block]

    @property
    def circuit(self) -> Circuit:
        return circuit_from_indices(self.n, self.indices, self.path)

    def __len__(self) -> int:
        return sum(len(block) for block in self.blocks)

    def __str__(self) -> str:
        if not self.blocks:
            return "(empty)"
        return " | ".join(" ".join(f"M{i}" for i in block) for block in self.blocks)


def canonical_blocks(n: int, perm: Sequence[int], path: Sequence[int]) -> List[List[int]]:
    """Algorithm 1 on index lists. Returns the blocks in execution order.

    perm[s] is the output state for input state s. The target arrangement is
    (f(a_0), f(a_1), ...). Position i is filled by moving the wanted string
    from its position l down to i with the gates M_i M_{i+1} ... M_{l-1}.
    """
    size = 1 << n
    if sorted(perm) != list(range(size)):
        raise ValueError("perm is not a permutation of the n-bit states")
    current = list(path)
    built: List[List[int]] = []              # construction order: C_0, C_1, ...
    for i in range(size):
        wanted = perm[path[i]]
        l = current.index(wanted, i)
        if l > i:
            built.append(list(range(i, l)))
            current.insert(i, current.pop(l))
    assert current == [perm[a] for a in path]
    return built[::-1]


def canonical_from_function(n: int, perm: Sequence[int],
                            path: Optional[Sequence[int]] = None) -> CanonicalForm:
    """Canonical circuit of the reversible function perm (Algorithm 1)."""
    path = _resolve_path(n, path)
    blocks = canonical_blocks(n, perm, path)
    return CanonicalForm(n, tuple(path), tuple(tuple(b) for b in blocks))


def canonical_form(circuit: Circuit, path: Optional[Sequence[int]] = None) -> CanonicalForm:
    """Canonical form of a circuit: simulate it, then apply Algorithm 1."""
    return canonical_from_function(circuit.n, circuit.permutation(), path)


def function_from_table(n: int, table: Dict[str, str]) -> List[int]:
    """Build perm from a truth table written with paper-style strings."""
    perm = [0] * (1 << n)
    for inp, out in table.items():
        perm[state_of(inp)] = state_of(out)
    if len(table) != 1 << n or sorted(perm) != list(range(1 << n)):
        raise ValueError("the table is not a complete reversible truth table")
    return perm

## Step 3: Verification script

In [ ]:
%%writefile verify_phase2.py
"""Phase 2 verification: canonical form of reversible circuits.

Checks, in order:

  A. The paper's worked examples (Examples 4, 5 and 7) are reproduced.
  B. Algorithm 1 on every 2-bit and 3-bit reversible function, and on random
     4-bit and 5-bit functions: the canonical circuit computes the function,
     satisfies Definition 1 and Fact 1, and different functions get
     different canonical circuits.
  C. Equivalent circuits get the same canonical form; non-equivalent
     circuits get different ones. Equivalent circuits are produced by
     rewriting with the Phase 1 rules.
  D. Other Hamiltonian paths also work (Remark 1 of the paper).
  E. Size of the canonical circuit as the number of lines grows.

Run:  python verify_phase2.py
"""
from __future__ import annotations

import random
import sys
import time
from itertools import permutations
from math import factorial

from revcircuit import CNOT, Circuit, X, random_circuit, random_gate
import rules as R
from canonical import (bits_of, canonical_blocks, canonical_form,
                       canonical_from_function, circuit_from_indices,
                       count_canonical_forms, delta_gates, fact1_holds,
                       function_from_table, gray_path, is_canonical,
                       is_hamiltonian_path, max_canonical_gates)

# The circuit drawings use Unicode symbols; make sure they print on Windows too.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8")

rng = random.Random(2026)
failures = 0


def report(label: str, ok: bool, detail: str = "") -> None:
    global failures
    failures += not ok
    print(f"  {label:<58}{'PASS' if ok else 'FAIL'}" + (f"  {detail}" if detail else ""))


# ---------------------------------------------------------------------------
# A. Worked examples from the paper
# ---------------------------------------------------------------------------
def part_a() -> None:
    print("A. Worked examples from the paper")
    n = 3
    path = gray_path(n)
    expected_path = "000 001 011 010 110 111 101 100".split()
    report("Hamiltonian path of Example 4",
           [bits_of(s, n) for s in path] == expected_path)

    # Example 4: function of Table I, canonical circuit M3 M4 M5 M0 M1 M2.
    table = {"000": "010", "001": "000", "010": "101", "011": "001",
             "100": "100", "101": "111", "110": "011", "111": "110"}
    f = function_from_table(n, table)
    cf4 = canonical_from_function(n, f)
    report("Example 4: canonical circuit of Table I",
           cf4.indices == [3, 4, 5, 0, 1, 2] and cf4.circuit.permutation() == tuple(f),
           str(cf4))

    # Example 5: a 9-gate circuit over Delta_H reduces to C2 C1 C0.
    c5 = circuit_from_indices(n, [4, 2, 3, 0, 4, 1, 3, 4, 0])
    cf5 = canonical_form(c5)
    report("Example 5: M4 M2 M3 M0 M4 M1 M3 M4 M0",
           cf5.indices == [2, 3, 1, 0, 1], str(cf5))

    # Example 7: the circuit of Example 3, CNOT[q0,q1] CNOT[q1,q2].
    c7 = Circuit(n, [CNOT(0, 1), CNOT(1, 2)])
    cf7 = canonical_form(c7)
    report("Example 7: CNOT[q0,q1] CNOT[q1,q2]",
           cf7.indices == [5, 6, 4, 5, 6, 2], str(cf7))

    print("\n  Gates of Delta_H for 3 lines (Fig. 4 of the paper):")
    for i, g in enumerate(delta_gates(n)):
        print(f"    M{i}: exchanges {bits_of(path[i], n)} and {bits_of(path[i + 1], n)}   {g}")
    print("\n  Canonical circuit of Example 4:  " + str(cf4))
    for line in cf4.circuit.draw().splitlines():
        print("    " + line)
    print()


# ---------------------------------------------------------------------------
# B. Algorithm 1 on reversible functions
# ---------------------------------------------------------------------------
def check_function(n: int, perm, path=None):
    """Return (canonical form, all properties hold)."""
    cf = canonical_from_function(n, perm, path)
    idx = cf.indices
    ok = (cf.circuit.permutation() == tuple(perm)
          and is_canonical(idx, n) and fact1_holds(idx, n)
          and len(idx) <= max_canonical_gates(n))
    return cf, ok


def part_b() -> dict:
    print("B. Algorithm 1 on reversible functions")
    print(f"  {'Lines':<7}{'Functions':>10}{'Coverage':>12}{'Distinct forms':>16}"
          f"{'Avg gates':>11}{'Max gates':>11}{'Bound':>8}  Result")
    stats = {}
    for n, sample in ((2, None), (3, None), (4, 2000), (5, 200)):
        size = 1 << n
        if sample is None:
            functions = permutations(range(size))
            coverage = "all"
        else:
            functions = (tuple(rng.sample(range(size), size)) for _ in range(sample))
            coverage = "random"
        seen, total_gates, max_gates, count, ok = {}, 0, 0, 0, True
        for perm in functions:
            cf, good = check_function(n, perm)
            ok &= good
            key = tuple(cf.indices)
            # Two different functions must never share a canonical circuit.
            ok &= seen.setdefault(key, perm) == perm
            total_gates += len(cf)
            max_gates = max(max_gates, len(cf))
            count += 1
        if sample is None:
            # Every function has its own form, and the number of index lists
            # allowed by Definition 1 equals the number of functions.
            ok &= len(seen) == count == factorial(size) == count_canonical_forms(n)
        stats[n] = (count, total_gates / count, max_gates)
        global failures
        failures += not ok
        print(f"  {n:<7}{count:>10}{coverage:>12}{len(seen):>16}{total_gates / count:>11.1f}"
              f"{max_gates:>11}{max_canonical_gates(n):>8}  {'PASS' if ok else 'FAIL'}")
    print("  Each row checks: the canonical circuit computes the function, satisfies")
    print("  Definition 1 and Fact 1, and no two functions share a canonical circuit.\n")
    return stats


# ---------------------------------------------------------------------------
# C. Equivalent circuits give the same canonical form
# ---------------------------------------------------------------------------
def scramble(c: Circuit, steps: int, max_len: int = 60) -> Circuit:
    """Rewrite a circuit with randomly chosen Phase 1 rules. The function is unchanged."""
    for _ in range(steps):
        i = rng.randrange(len(c) + 1)
        move = rng.randrange(12)
        new = None
        if move == 0 and len(c) + 2 <= max_len:
            new = R.rule1_insert(c, i, random_gate(c.n, rng))
        elif move == 1 and len(c) + 1 <= max_len:
            new = R.rule2_split(c, i, rng.randrange(c.n))
        elif move == 2 and len(c) + 2 * c.n <= max_len:
            new = R.rule8_expand(c, i)
        else:
            fn = (R.rule1_cancel, R.rule2_merge, R.rule3_commute, R.rule4_apply,
                  R.rule5_apply, R.rule7_commute, R.rule8_collapse, R.rule9_apply,
                  R.rule10_apply)[move - 3]
            new = fn(c, i)
        if new is not None:
            c = new
    return c


def part_c() -> None:
    print("C. Equivalent and non-equivalent circuits")
    print(f"  {'Lines':<7}{'Pairs':>7}{'Rewritten differently':>23}"
          f"{'Same form':>11}{'Changed circuit differs':>25}  Result")
    for n, pairs in ((3, 300), (4, 200), (5, 100)):
        differ = same = separated = 0
        for _ in range(pairs):
            c = random_circuit(n, rng.randint(3, 10), rng)
            d = scramble(c, 40)
            differ += c != d
            same += canonical_form(c).indices == canonical_form(d).indices
            # One extra gate changes the function, so the form must change.
            k = rng.randrange(len(d) + 1)
            e = d.replace(k, k, [random_gate(n, rng)])
            separated += canonical_form(e).indices != canonical_form(c).indices
        ok = same == pairs and separated == pairs and differ > pairs // 2
        global failures
        failures += not ok
        print(f"  {n:<7}{pairs:>7}{differ:>23}{same:>11}{separated:>25}  "
              f"{'PASS' if ok else 'FAIL'}")
    print("  Each pair is a random circuit and a copy rewritten by 40 random rule steps.\n")


# ---------------------------------------------------------------------------
# D. Other Hamiltonian paths (Remark 1)
# ---------------------------------------------------------------------------
def random_path(n: int):
    """A Hamiltonian path made from the Gray code by renaming lines, flipping bits, reversing."""
    order = list(range(n))
    rng.shuffle(order)
    mask = rng.randrange(1 << n)
    path = [sum(((s >> i) & 1) << order[i] for i in range(n)) ^ mask for s in gray_path(n)]
    return path[::-1] if rng.random() < 0.5 else path


def part_d() -> None:
    print("D. Other Hamiltonian paths")
    for n, paths, per_path in ((3, 20, 200), (4, 20, 50)):
        ok, used = True, set()
        for _ in range(paths):
            path = random_path(n)
            ok &= is_hamiltonian_path(n, path)
            used.add(tuple(path))
            for _ in range(per_path):
                perm = tuple(rng.sample(range(1 << n), 1 << n))
                ok &= check_function(n, perm, path)[1]
        report(f"{n} lines: {len(used)} different paths, {per_path} functions each", ok)
    print()


# ---------------------------------------------------------------------------
# E. Growth of the canonical circuit
# ---------------------------------------------------------------------------
def part_e(stats: dict) -> None:
    print("E. Size of the canonical circuit")
    print("  Canonical circuit of ONE gate placed on n lines (Gray-code path).")
    print("  Every canonical gate is an n-bit gate with n - 1 controls.")
    print(f"  {'Lines':<7}{'X[q0]':>12}{'CNOT[q0,q1]':>14}{'Worst case':>13}")
    for n in range(2, 11):
        path = gray_path(n)
        sizes = []
        for gate in (X(0), CNOT(0, 1)):
            perm = Circuit(n, [gate]).permutation()
            sizes.append(sum(len(b) for b in canonical_blocks(n, perm, path)))
        print(f"  {n:<7}{sizes[0]:>12}{sizes[1]:>14}{max_canonical_gates(n):>13}")
    print("  Worst case = 2^n (2^n - 1) / 2 gates.")
    print("  Average over the functions tested in part B: "
          + ", ".join(f"{n} lines {avg:.1f}" for n, (_, avg, _) in stats.items()) + ".\n")


def main() -> int:
    start = time.time()
    print("Phase 2 verification: canonical form\n")
    part_a()
    stats = part_b()
    part_c()
    part_d()
    part_e(stats)
    print(f"Time: {time.time() - start:.1f} s")
    print("\nAll checks passed." if failures == 0 else f"\n{failures} check(s) FAILED.")
    return 1 if failures else 0


if __name__ == "__main__":
    sys.exit(main())

## Step 4: Run the verification

Every line should show **PASS**, and the output should end with `All checks passed.`

In [ ]:
!python verify_phase2.py

## Step 5: Try it yourself

Two circuits that look different but compute the same function get the same canonical form.
A circuit that computes a different function gets a different one.

Canonical circuits are printed as gate names, with `|` separating the blocks.
Drawing symbols: `⊕` target, `●` positive control, `○` negative control.

In [ ]:
import importlib
import revcircuit, rules, canonical
for m in (revcircuit, rules, canonical):
    importlib.reload(m)

from revcircuit import Circuit, G, X, CNOT, TOF
from canonical import canonical_form

# Circuits A and B are the two sides of Example 2 in the paper, on 3 lines.
a = Circuit(3, [X(0), CNOT(0, 1), X(0)])
b = Circuit(3, [G([], [0], 1)])
# Circuit C is a plain CNOT: a different function.
c = Circuit(3, [CNOT(0, 1)])

for name, circ in (("A", a), ("B", b), ("C", c)):
    form = canonical_form(circ)
    print(f"Circuit {name}: {circ}")
    print(circ.draw())
    print(f"Canonical form ({len(form)} gates): {form}")
    print()

print("A and B equivalent?         ", a.equivalent(b))
print("A and B same canonical form?", canonical_form(a).indices == canonical_form(b).indices)
print("A and C same canonical form?", canonical_form(a).indices == canonical_form(c).indices)
print()
print("Canonical circuit of A, drawn:")
print(canonical_form(a).circuit.draw())